# 初识 CANNBot——让 AI 帮你写 NPU 算子

本节介绍 CANNBot 的基础概念。CANNBot 是一支专门为昇腾 NPU 开发而组建的 AI 团队。本节围绕三个问题展开，学习时长约 10 分钟：

1. **编程 Agent 和聊天 AI 有什么区别？**
2. **CANNBot 是什么、怎么组织的？**（三层架构与工作流）
3. **它背后是一个怎样的生态？**（六仓全景一图流）

本节不涉及安装与实操，下一节将亲手让 CANNBot 交付第一个算子。

> **前置要求**：具备 NPU/CANN 基础认知（学过[快速上手](https://gitcode.com/cann/cann-learning-hub/tree/master/quick_start)或 [Ascend C 教程（Kernel 直调）第 1 章](https://gitcode.com/cann/cann-learning-hub/blob/master/tutorials/ascendc_operator_development_light/01_basic_overview/01.03_cann_arch_ascend_npu_principle.ipynb)水平即可）。

## 为什么需要编程 Agent

假设要给昇腾 NPU 开发一个 `add`（矢量加法）算子。开发者通常用过 ChatGPT、DeepSeek 这类 AI 聊天助手，但它们只能给出建议，无法代替人工完成开发：算子要自己写、测试要自己跑、报错要自己修。编程 Agent 弥补了这一空白，它能直接完成上述全部工作。

**CANNBot** 就是为这一目标设计的：一支专门为昇腾 NPU 开发而组建的 AI 团队。

## 1. 聊天 AI 与编程 Agent 的区别

以开发 `add`（矢量加法）算子为例，分别请两者协助，行为差异如下：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">对比维度</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">聊天 AI（顾问）</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">编程 Agent（员工）</th></tr></thead><tbody>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">能看什么</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">仅能看到粘贴进对话框的内容</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">主动读你的工程目录、配置、已有代码</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">能做什么</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">输出文字和代码片段，需人工复制粘贴</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">创建/修改文件、执行命令、编译、跑测试</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">怎么确认结果</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">结果是否正确需人工验证</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">自行运行测试、比对结果，以通过/失败作为结论</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">交付物</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">一段聊天记录</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">一整套可编译工程：代码、测试、文档</td></tr>
</tbody></table>

本质区别是**闭环**：

```text
理解需求 → 写代码 → 执行（编译/运行）→ 看结果 → 改代码 → 再执行 → …… → 验证通过，交付
```

聊天 AI 的流程到给出代码为止，后续环节全部依赖人工；编程 Agent 将整个闭环掌握在自己手中。

### 三个关键词

- **Agent（智能体）**＝大模型（大脑）＋工具（手脚：读写文件、执行命令）＋自主决策（开发者给定目标，Agent 自行规划步骤）。承载 Agent 运行的软件叫**宿主**，如 OpenCode、Claude Code、Cursor；
- **Skill（技能）**＝写给 Agent 看的领域知识手册。Agent 不会天生掌握 Ascend C 的 Tiling 切分方法。将正确做法写成手册、执行任务前按需加载，可以显著降低出错率。Skill 人人可写、越积越厚，这是社区共建的核心；
- **多 Agent 协作**＝一支团队而不是一个人。PM 只负责调度，架构师设计、开发者编码、测试者验收。**编写代码的角色不能自行宣布测试通过**，与人类工程的开发/测试分离原则同理。

> ⚠️ **重要提醒**：Agent 生成的代码受模型、语料、指令影响，**不能保证完全正确**，必须经过独立验证才能使用。CANNBot 因此将**独立验收**设计为工作流的基本原则。

## 2. CANNBot 是什么

CANNBot 的定位：

> **CANNBot 是面向 CANN 开发的系列智能体，把昇腾开发中知识密集、流程标准的环节交给 AI 团队自动完成，使开发者聚焦于决策本身。**

它能覆盖的场景以算子开发为主线，辐射四大块：**Ascend C 算子开发**（Kernel 直调/注册算子，典型指令：帮我开发一个 add 算子，float32）、**多语言路径**（PyPTO / TileLang / Triton 等替代编程范式（各自定位详见 01.03 §3 五条路径））、**推理与性能**（模型推理优化、算子性能演进）、**迁移与配套**（CUDA→Ascend 迁移、质量保障）。目前全生态已沉淀 **200+ 款 Skill**。

同样重要的是它的边界：**不保证代码完全正确**（因此必须独立验收）、**不替开发者理解需求**（需求越含糊，它发挥越自由）、**不是黑盒**（每一步留有记录，开发者始终是管理者）。

### 三层架构：菜谱、厨师与手册

> **Plugin 像一道菜的完整菜谱，Agent 是按岗位分工的厨师，Skill 是岗位手边的知识手册。**

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">层次</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">回答的问题</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">例子</th></tr></thead><tbody>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Plugin（应用编排层）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">按什么流程干</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">ops-direct-invoke-flash（本课程实操用）</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Agent（角色执行层）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">谁来干这一步</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">architect / developer / reviewer</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Skill（知识能力层）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">怎么干才对</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">ascendc-api-best-practices、tiling-design</td></tr>
</tbody></table>

这个分层的关键好处是：**新增一个场景只需新增一份菜谱（Plugin），厨师和手册直接复用**。这也是几十个场景能够快速铺开的结构原因。

<img src="./images/01.01_three_layer_architecture.png" alt="CANNBot 三层架构示意图" width="700px">


### 工作流：轻量版与完整版

本课程实操用的 **Flash 轻量版**（ops-direct-invoke-flash）走八个阶段的 TDD 主线（TDD = 测试驱动开发：先写测试、后写实现）：**环境探测与骨架 → 定义/设计文档 → 测试套件 → 核函数实现 → 真机验证 → 收尾**。需要更强过程管控的场景（正式交付、多人协作）可用完整版（ops-direct-invoke）：PM + 三角色（architect/developer/verifier），七节点工作流 + **节点级验收对偶**：每个节点由验收者独立核证据，失败自动重试。两版共享同一套原则：**AI 产出必须被独立验证**。（完整版的机制设计将在[进阶篇 02.01 节](../02_core_workflows/02.01_complete_workflow.ipynb)深入拆解。）


## 3. 用代码看清：一次开发的完整链路

运行下面的代码，画出从提出需求、Agent 团队分工到交付验证过的算子的完整链路（本代码随处可跑，无需 NPU）：

In [ ]:
def draw_cannbot_pipeline():
    """画出一条需求在 CANNBot（Flash 版）中的完整流转。"""
    bar = "=" * 60
    print(f"{bar}")
    print("你说：「开发一个 add_custom 算子：z = x + y，float32，(8, 2048)」")
    print(f"{bar}")
    stages = [
        ("环境探测", "识别芯片（如 Ascend910B3），判定复杂度档位"),
        ("定义文档", "数学语义 / dtype 策略 / 边界场景"),
        ("设计文档", "tiling 方案 / UB 预算 / 校验记录"),
        ("测试先行", "参数化用例矩阵 + 边界值，先写好后验证"),
        ("内核实现", "CopyIn → 计算 → CopyOut 三级流水 + tiling"),
        ("真机验证", "编译 → pytest 在真实 NPU 上与 CPU 参考逐值比对"),
        ("收尾归档", "全绿状态 + 交付报告"),
    ]
    for name, desc in stages:
        print(f"  [{name}]  {desc}")
    print(f"{bar}")
    print("交付：operators/add_custom/{add_custom.asc, test_add_custom.py, CMakeLists.txt, docs/}")
    print()
    print("要点：文档先行、测试先行、真机验证——AI 产出必须被独立验证。")
    print("      复杂算子还会追加多 Agent 评审与测试门禁（图为 simple 档快速路径）。")

draw_cannbot_pipeline()

## 4. 全景一图：CANNBot 背后的六仓生态

CANNBot 不只是一个仓库，而是一组分工明确的仓库群（课程中直接接触的是前两个）:

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">仓库</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">一句话定位</th></tr></thead><tbody>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">cannbot（应用仓）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">工作流编排 × 场景 Plugin，按场景组织的应用商店</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">cannbot-skills</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">知识手册总库（200+ 款 Skill，覆盖算子开发/模型优化/图优化/迁移/测试/运维等全域），下一节将克隆此仓库</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">cannbot-knowledge</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">更新较快的知识库（API 语义/错误码/芯片规格），使用时按需查询</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">cannbot-dsl</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">为 Agent 重设计的算子语言：易错的硬件表达交给编译器</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">cann-bench</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">算子能力公共标尺（评测 + 排行榜），01.03 节详述</td></tr>
<tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">cannbot-sentry</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">观测 × 审计：让 Agent 的执行全过程可观测、可审计</td></tr>
</tbody></table>

现阶段只需关注两点：安装并使用 skills 仓（下一节完成安装），遇到问题去应用仓或 skills 仓提 Issue（01.03 节介绍路径）。其余四仓了解分工即可。

<img src="./images/01.01_six_repos.png" alt="CANNBot 六仓生态示意图" width="700px">

此外，每款 Skill 都按 [Agent Skills 开放标准](https://agentskills.io) 封装，**同一套知识可装进 8 类以上宿主**（OpenCode、Claude Code、Trae、Cursor 等）。无论使用哪种工具，都能接入 CANNBot。

## 5. 带走物

| 类别 | 内容 |
|---|---|
| **一句话** | CANNBot 是一支做 NPU 算子开发的 AI 团队：Plugin 编排、Agent 执行、Skill 供知识 |
| **认知框架** | 三层架构 + 工作流两版选型 + 六仓生态全景 |
| **下一步** | [01.02](./01.02_flash_workflow.ipynb)：亲手让它交付第一个算子 |

## 6. 课后练习

> 练习答案：[`answer/01.01_answer.txt`](./answer/01.01_answer.txt)


本节介绍了编程 Agent 与 CANNBot 的基本概念，请根据学习内容完成以下题目自测。

1. （单选题）编程 Agent 与聊天 AI 最本质的区别是？  
    A. 编程 Agent 用的大模型参数量更大  
    B. 编程 Agent 能执行写代码→运行→看结果→修正的完整闭环，聊天 AI 到给出回答就停止  
    C. 编程 Agent 只能回答编程问题  
    D. 聊天 AI 不能生成代码

2. （多选题）关于 Skill（技能），下列说法正确的有哪些？  
    A. Skill 是写给 Agent 看的领域知识手册，执行任务前按需加载  
    B. Skill 只能由官方团队编写，社区开发者不能贡献  
    C. CANNBot 的 Skill 遵循 Agent Skills 开放标准，可装进多种宿主  
    D. 把所有 Skill 一次性全部塞给模型是最佳实践

3. （单选题）按照 Plugin 像菜谱、Agent 像厨师、Skill 像手册的类比，新增一个开发场景主要需要新增什么？  
    A. 新增一批 Agent（厨师）  
    B. 新增一个 Plugin（菜谱），复用现有 Agent 与 Skill  
    C. 重写所有 Skill  
    D. 更换底层大模型

4. （单选题）Agent 生成的代码不能保证完全正确。面对这一局限，CANNBot 的应对方式是？  
    A. 限制能力，只允许生成简单代码，复杂算子必须由人编写  
    B. 将独立验收设计进工作流：测试先行、真机验证、验收证据可复现  
    C. 在输出中提示用户自行承担全部风险  
    D. 依赖大模型能力提升，待未来版本自然解决

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/01.01_answer.txt

## 7. 要点回顾

- **闭环是分界线**：聊天 AI 只给建议，编程 Agent 直接执行（读写文件、执行命令、自我验证）；
- **三个关键词**：Agent（大脑+手脚+自主决策）、Skill（按需加载的手册，社区共建）、多 Agent 协作（执行与验收分离）；
- **CANNBot**：面向 CANN 开发的智能体系，三层架构（Plugin 编排 / Agent 执行 / Skill 供能），Flash 轻量版八阶段工作流；
- **六仓生态**：使用 skills 仓、遇到问题提 Issue，其余各司其职；Skill 开放标准使其在任何宿主中都能使用；

概念基础到此建立。下一节进入实操：[第一个算子——从安装到验收的完整体验](./01.02_flash_workflow.ipynb)，亲手让 CANNBot 交付第一个算子。